/**
 * ETL-КОНВЕЄР: Консолідація та очищення історичних реєстрів МВС (2023–2025)
 * * ПРИЗНАЧЕННЯ: Автоматизація процесів Extract, Transform, Load. Читання сирих 
 * текстових логів у форматі CSV, валідація аномалій, нормалізація даних 
 * та пакетна матеріалізація у реляційну СУБД SQLite.
 *
 * ТЕХНІЧНИЙ СТЕК: Python (Pandas, SQLAlchemy).
 * ОПТИМІЗАЦІЯ: Використання параметра chunksize=50000 для контролю використання 
 * оперативної пам'яті (RAM) при роботі з великими датасетами (~6.69 млн рядків сумарно).
 */

In [1]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt

In [2]:
# Створюємо підключення до центрального сховища даних
engine = create_engine('sqlite:///d:/data_sets/project_transport/transport_main_db.db')

# Визначення типів даних для запобігання втрати бізнес-логіки
# (наприклад, перетворення кодів КОАТУУ на експоненціальний запис float)
column_types = {
    'REG_ADDR_KOATUU': str,  
    'VIN': str,              
    'OPER_CODE': str,        
    'OWN_WEIGHT': str,       
    'TOTAL_WEIGHT': str,     
    'DEP_CODE': str         
}

# =================================================================
# БЛОК 1: ОБРОБКА ДАНИХ ЗА 2025 РІК
# =================================================================
print("--- Запуск процесу для реєстру 2025 року ---")

# Завантаження файлу 2025 року (Extract)
public2025 = pd.read_csv(
    "d:/data_sets/project_transport/reestrtz31122025.csv", 
    sep=';', 
    dtype=column_types,
    low_memory=False
)

# Трансформація даних (Data Transformation)
# Перетворення ваги та дат до системних типів даних
public2025['OWN_WEIGHT'] = pd.to_numeric(public2025['OWN_WEIGHT'].str.replace(',', '.'), errors='coerce')
public2025['TOTAL_WEIGHT'] = pd.to_numeric(public2025['TOTAL_WEIGHT'].str.replace(',', '.'), errors='coerce')
public2025['D_REG'] = pd.to_datetime(public2025['D_REG'], format='%d.%m.%y', errors='coerce')

# Очищаємо бренди від випадкових пробілів та уніфікуємо регістр (Data Cleaning)
public2025['BRAND'] = public2025['BRAND'].str.upper().str.strip()

# Очищення текстового шуму та валідація часових меж року випуску ТЗ
public2025['MAKE_YEAR'] = pd.to_numeric(public2025['MAKE_YEAR'], errors='coerce')
public2025 = public2025[(public2025['MAKE_YEAR'] >= 1950) & (public2025['MAKE_YEAR'] <= 2026)]

# Очищення критичних пустих значень (Data Quality Filter)
public2025 = public2025.dropna(subset=['D_REG', 'BRAND'])

print("Дані за 2025 рік успішно завантажені та очищені!")
print(f"Кількість рядків за 2025 рік: {len(public2025)}")

# Матеріалізація в базі даних (Load)
print("Завантажую 2025 рік в SQL...")
public2025.to_sql(
    name='reestr_2025', 
    con=engine, 
    if_exists='replace', # Якщо таблиця вже існувала — перестворюємо її чистою
    index=False, 
    chunksize=50000
)

print("🎉 База даних створена! Таблиця 'reestr_2025' успішно додана в базу даних!")

--- Запуск процесу для реєстру 2025 року ---
Дані за 2025 рік успішно завантажені та очищені!
Кількість рядків за 2025 рік: 2229842
Завантажую 2025 рік в SQL...
🎉 База даних створена! Таблиця 'reestr_2025' успішно додана в базу даних!


In [3]:
# Валідація результату очищення перед завантаженням (Data Quality Check)
print("Приклад очищених даних для перевірки:")
print(public2025[['BRAND', 'MAKE_YEAR', 'D_REG', 'OWN_WEIGHT']].head(3))

Приклад очищених даних для перевірки:
   BRAND  MAKE_YEAR      D_REG  OWN_WEIGHT
0  LEXUS       2008 2025-01-01      2075.0
1     MG       2014 2025-01-01      1265.0
2  SKODA       2007 2025-01-01      1593.0


In [4]:
# =================================================================
# БЛОК 2: ОБРОБКА ДАНИХ ЗА 2024 РІК
# =================================================================
print("--- Запуск процесу для реєстру 2024 року ---")

# Завантаження файлу 2024 року (Extract)
public2024 = pd.read_csv(
    "d:/data_sets/project_transport/reestrtz31122024.csv", 
    sep=';', 
    dtype=column_types,
    low_memory=False
)

# Трансформація даних (Data Transformation)
# Перетворення ваги та дат до системних типів даних
public2024['OWN_WEIGHT'] = pd.to_numeric(public2024['OWN_WEIGHT'].str.replace(',', '.'), errors='coerce')
public2024['TOTAL_WEIGHT'] = pd.to_numeric(public2024['TOTAL_WEIGHT'].str.replace(',', '.'), errors='coerce')
public2024['D_REG'] = pd.to_datetime(public2024['D_REG'], format='%d.%m.%y', errors='coerce')

# Очищаємо бренди від випадкових пробілів та уніфікуємо регістр (Data Cleaning)
public2024['BRAND'] = public2024['BRAND'].str.upper().str.strip()

# Очищення текстового шуму та валідація часових меж року випуску ТЗ
public2024['MAKE_YEAR'] = pd.to_numeric(public2024['MAKE_YEAR'], errors='coerce')
public2024 = public2024[(public2024['MAKE_YEAR'] >= 1950) & (public2024['MAKE_YEAR'] <= 2026)]

# Очищення критичних пустих значень (Data Quality Filter)
public2024 = public2024.dropna(subset=['D_REG', 'BRAND'])

print("Дані за 2024 рік успішно завантажені та очищені!")
print(f"Кількість рядків за 2024 рік: {len(public2024)}")

# Матеріалізація в базі даних (Load)
print("Завантажую 2024 рік в SQL...")
public2024.to_sql(
    name='reestr_2024', 
    con=engine, 
    if_exists='replace', 
    index=False, 
    chunksize=50000
)

print("🎉 Таблиця 'reestr_2024' успішно додана в базу даних!\n")

--- Запуск процесу для реєстру 2024 року ---
Дані за 2024 рік успішно завантажені та очищені!
Кількість рядків за 2024 рік: 2344450
Завантажую 2024 рік в SQL...
🎉 Таблиця 'reestr_2024' успішно додана в базу даних!



In [5]:
# Валідація результату очищення перед завантаженням (Data Quality Check)
print("Приклад очищених даних для перевірки:")
print(public2024[['BRAND', 'MAKE_YEAR', 'D_REG', 'OWN_WEIGHT']].head(3))

Приклад очищених даних для перевірки:
        BRAND  MAKE_YEAR      D_REG  OWN_WEIGHT
0      ATAMAN       2023 2024-01-01      5840.0
1      TOYOTA       2010 2024-01-01      2715.0
2  VOLKSWAGEN       2002 2024-01-01      1204.0


In [6]:
# =================================================================
# БЛОК 3: ОБРОБКА ДАНИХ ЗА 2023 РІК
# =================================================================
print("--- Запуск процесу для реєстру 2023 року ---")

# Завантаження файлу 2023 року (Extract)
public2023 = pd.read_csv(
    "d:/data_sets/project_transport/reestrtz31122023.csv", 
    sep=';', 
    dtype=column_types,
    low_memory=False
)

# Трансформація даних (Data Transformation)
# Перетворення ваги та дат до системних типів даних
public2023['OWN_WEIGHT'] = pd.to_numeric(public2023['OWN_WEIGHT'].str.replace(',', '.'), errors='coerce')
public2023['TOTAL_WEIGHT'] = pd.to_numeric(public2023['TOTAL_WEIGHT'].str.replace(',', '.'), errors='coerce')
public2023['D_REG'] = pd.to_datetime(public2023['D_REG'], format='%d.%m.%y', errors='coerce')

# Очищаємо бренди від випадкових пробілів та уніфікуємо регістр (Data Cleaning)
public2023['BRAND'] = public2023['BRAND'].str.upper().str.strip()

# Очищення текстового шуму та валідація часових меж року випуску ТЗ
public2023['MAKE_YEAR'] = pd.to_numeric(public2023['MAKE_YEAR'], errors='coerce')
public2023 = public2023[(public2023['MAKE_YEAR'] >= 1950) & (public2023['MAKE_YEAR'] <= 2026)]

# Очищення критичних пустих значень (Data Quality Filter)
public2023 = public2023.dropna(subset=['D_REG', 'BRAND'])

print("Дані за 2023 рік успішно завантажені та очищені!")
print(f"Кількість рядків за 2023 рік: {len(public2023)}")

# Матеріалізація в базі даних (Load)
print("Завантажую 2023 рік в SQL...")
public2023.to_sql(
    name='reestr_2023', 
    con=engine, 
    if_exists='replace', 
    index=False, 
    chunksize=50000
)

print("🎉 Таблиця 'reestr_2023' успішно додана в базу даних!\n")
print("🏆 ЕТАП ЕТL ПОВНІСТЮ ЗАВЕРШЕНО. База даних сформована та готова до роботи!")

--- Запуск процесу для реєстру 2023 року ---
Дані за 2023 рік успішно завантажені та очищені!
Кількість рядків за 2023 рік: 2124651
Завантажую 2023 рік в SQL...
🎉 Таблиця 'reestr_2023' успішно додана в базу даних!

🏆 ЕТАП ЕТL ПОВНІСТЮ ЗАВЕРШЕНО. База даних сформована та готова до роботи!


In [7]:
# Валідація результату очищення перед завантаженням (Data Quality Check)
print("Приклад очищених даних для перевірки:")
print(public2023[['BRAND', 'MAKE_YEAR', 'D_REG', 'OWN_WEIGHT']].head(3))

Приклад очищених даних для перевірки:
    BRAND  MAKE_YEAR      D_REG  OWN_WEIGHT
0    FORD       2012 2023-04-28      1283.0
1   LEXUS       2010 2023-07-07      1895.0
2  NISSAN       2012 2023-06-03      1167.0


In [8]:
# =================================================================
# ЕТАП САМОПЕРЕВІРКИ (ТІЛЬКИ ДЛЯ СТВОРЕНИХ РЕЄСТРІВ)
# =================================================================
print("\n🔍 Запуск фінальної перевірки бази даних...")

# Явно вказуємо список таблиць, які були згенеровані цим ETL-скриптом
target_tables = ['reestr_2023', 'reestr_2024', 'reestr_2025']

# Швидкий підрахунок рядків у кожній цільовій таблиці через SQL-запити
for table in target_tables:
    try:
        row_count = pd.read_sql(f"SELECT COUNT(*) FROM {table}", con=engine).iloc[0, 0]
        print(f"📊 Таблиця '{table}': {row_count:,} рядків завантажено.")
    except Exception as e:
        print(f"❌ Помилка при перевірці таблиці '{table}': {e}")

print("\n🚀 Усе супер! Перевірку завантаження реєстрів завершено успішно.")


🔍 Запуск фінальної перевірки бази даних...
📊 Таблиця 'reestr_2023': 2,124,651 рядків завантажено.
📊 Таблиця 'reestr_2024': 2,344,450 рядків завантажено.
📊 Таблиця 'reestr_2025': 2,229,842 рядків завантажено.

🚀 Усе супер! Перевірку завантаження реєстрів завершено успішно.
